# Démo 2 · D’une API à une table pandas

Dans la démo 1, nous avons commencé avec des tables prêtes à utiliser. Cette fois,
nous allons **récupérer les données d’un vrai match de la NHL sur Internet et
construire la table nous-mêmes**.
Aucune expérience des API ni connaissance du hockey n’est nécessaire. Vous devriez
être à l’aise pour exécuter des cellules de notebook et reconnaître les listes et
les dictionnaires Python.

À la fin, vous serez capables de :

- Expliquer ce que sont une requête et une réponse d’API.
- Télécharger les données événement par événement d’un match avec Python.
- Sauvegarder les données originales et les recharger sans utiliser Internet.
- Transformer du JSON imbriqué en DataFrame pandas, l’inspecter et conserver les tirs et les buts.
- Vérifier les valeurs manquantes et les identifiants des événements avant d’utiliser la table.

**Notre pipeline :** requête → sauvegarde du JSON brut → chargement du JSON → création d’une table → vérification.

## Configuration : choisir l’environnement local ou Colab

**En local :** lancez `uv sync` à la racine du dépôt `ift3700-6758`, puis sélectionnez le noyau de l’environnement commun `.venv` sous Python 3.11. Consultez le [README du dépôt](../../../README.md) pour la configuration. Sautez la cellule d’installation facultative : elle n’installe rien en local.

**Google Colab | sans clonage du dépôt ni installation manuelle de uv :**

1. Ouvrez ce notebook dans Colab et choisissez un runtime **CPU**. L’installation nécessite Internet.
2. Dans la **cellule facultative ci-dessous**, définissez `INSTALL_COLAB_PACKAGES = True` et exécutez-la une fois. La cellule installe `uv`, puis l’utilise pour installer les bibliothèques dans le Python actuel du notebook. `sys.executable` est le chemin de ce Python; `subprocess.check_call` lance une commande et s’arrête si elle échoue.
3. Si Colab demande un redémarrage après l’installation, choisissez **Runtime → Restart session**.
4. Remettez le paramètre à `False`, puis exécutez les cellules de haut en bas avec **Maj+Entrée**. Répétez l’installation lorsque Colab vous attribue un nouveau runtime; un simple redémarrage de session conserve les paquets installés.

Ce notebook utilise **requests** pour les requêtes HTTP et **pandas** pour les tables; `json` et `pathlib` sont inclus dans Python.

La première exécution nécessite Internet pour télécharger le match. Les suivantes réutilisent le fichier JSON sauvegardé.


In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "pandas>=2.2,<4",
        "requests>=2.31,<3",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")


In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

## A. Qu’est-ce qu’une API ?

Une **API** (Application Programming Interface, ou interface de programmation)
permet à un programme de demander quelque chose à un autre programme selon des
règles convenues. Un site web renvoie souvent une page destinée à une personne;
une API web peut renvoyer des données structurées destinées à un programme.

Ici, votre notebook Python est le **client**, et la NHL exploite le **serveur**.
Notre requête demande : « Veuillez envoyer les événements enregistrés pour ce match. »

```text
Votre notebook  ── HTTP GET + une URL ──>  Serveur de la NHL
Votre notebook  <── statut + en-têtes + corps JSON ──  Serveur de la NHL
```

**HTTP** est le protocole utilisé pour ces messages. **GET** signifie récupérer
une ressource. **REST** est un style de conception d’API qui identifie les ressources
par des URL et utilise les méthodes HTTP pour interagir avec elles. Chaque requête
contient ce dont le serveur a besoin pour la traiter; le serveur n’a pas à se
souvenir d’une conversation précédente avec le client.
Nous n’avons besoin que de GET dans cette démo. Les API peuvent renvoyer d’autres
formats que JSON, et toutes les API ne sont pas des API REST.

Ce point d’accès public de la NHL ne nécessite actuellement **ni compte ni clé
API**. D’autres API peuvent exiger une authentification. Nous récupérons des
données, sans extraire le contenu d’une page HTML.

### Lire l’adresse avant d’écrire du code

Nous utiliserons le match de Montréal en Caroline du **21 mai 2026**. Un **point
d’accès** (*endpoint*) est une adresse de l’API qui donne accès à une ressource précise.

```text
https://api-web.nhle.com/v1/gamecenter/2025030311/play-by-play
        └── serveur ──┘ └──────── chemin du point d’accès ─┘
```

| Partie de l’URL | Signification ici |
|---|---|
| `https://` | HTTP sur une connexion chiffrée |
| `api-web.nhle.com` | Le serveur contacté |
| `/v1` | La version de l’API dans le chemin |
| `/gamecenter/2025030311` | La ressource correspondant au match et son identifiant |
| `/play-by-play` | Les données des événements que nous voulons |

`2025030311` est un **identifiant** de match, pas une date : `2025` correspond à
l’année de début de la saison, `03` aux séries éliminatoires, et `0311` identifie
le match de séries. Nous utilisons un identifiant connu; ne supposez pas que
chaque nombre possible correspond à un match.

Ouvrez la [réponse brute de l’API](https://api-web.nhle.com/v1/gamecenter/2025030311/play-by-play)
dans un navigateur et comparez-la avec la
[page du match sur le site de la NHL](https://www.nhl.com/gamecenter/mtl-vs-car/2026/05/21/2025030311/playbyplay).
Votre navigateur peut afficher le JSON sur une longue ligne ou sous forme d’arbre
que vous pouvez déplier.

**Anticipez :** quelle partie de l’URL changeriez-vous pour demander un autre match ?
Faudrait-il changer l’adresse du serveur ?

In [ ]:
game_id = 2025030311
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
print(url)

## B. Télécharger une seule fois et sauvegarder la réponse brute

Une réponse comprend un **code de statut**, des **en-têtes** qui la décrivent et
un **corps** qui contient les données. Pour une requête réussie ici, on s’attend
au statut `200` et au type de contenu `application/json`.

| Code | Signification | Que faire ? |
|---|---|---|
| `200` | Requête réussie | Inspecter les données renvoyées |
| `404` | Ressource introuvable | Vérifier le point d’accès et l’identifiant du match |
| `403` | Accès refusé | Vérifier les restrictions d’accès; demander à la personne qui enseigne au besoin |
| `429` | Trop de requêtes | Arrêter les requêtes et attendre; respecter `Retry-After` s’il est fourni |
| `5xx` | Problème du serveur | Conserver les données locales et réessayer plus tard |

Les deux cellules suivantes conservent les téléchargements dans un dossier local.
Un **cache** est une copie sauvegardée que l’on peut réutiliser. Les chemins
ci-dessous sont relatifs au répertoire de travail du noyau du notebook, qui peut
différer selon l’éditeur. Nous affichons le chemin complet pour le rendre visible.
Gardez le même répertoire de travail lorsque vous relancez le notebook.

In [ ]:
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
raw_path = raw_dir / f"{game_id}.json"
print("Raw data file:", raw_path.resolve())

Lisez la cellule de téléchargement avant de l’exécuter :

1. `exists()` vérifie si nous avons déjà sauvegardé ce match.
2. `requests.get(...)` envoie la requête GET. `timeout=30` limite l’attente pour
   établir la connexion et recevoir des données; ce n’est pas une limite sur la
   durée totale du téléchargement.
3. `raise_for_status()` arrête l’exécution en cas d’erreur HTTP. Une réponse
   d’erreur peut aussi contenir du JSON : réussir à le décoder ne prouve donc
   **pas** que la requête a réussi.
4. `.json()` décode la réponse en objets Python.
5. `json.dumps(...)` convertit ces objets en texte JSON; `write_text(...)` le sauvegarde.

Nous conservons les champs d’origine plutôt que de sauvegarder seulement une table
filtrée. Cela permet de modifier nos choix de nettoyage plus tard sans télécharger
à nouveau les données. L’indentation rend le JSON sauvegardé lisible; ses espaces
n’ont pas à être identiques à ceux de la réponse HTTP.

In [ ]:
if raw_path.exists():
    print("Already downloaded — reusing", raw_path.name)
else:
    response = requests.get(url, timeout=30)
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    response.raise_for_status()
    downloaded_game = response.json()

    # Check we received the requested game's event data before saving it.
    assert downloaded_game["id"] == game_id
    assert isinstance(downloaded_game["plays"], list)
    raw_path.write_text(json.dumps(downloaded_game, indent=2), encoding="utf-8")
    print("Saved", raw_path.name)

## C. Charger et explorer le JSON

Le chargement est une étape distincte : il lit un fichier sur votre ordinateur et
ne fait **aucune requête à l’API**. `read_text()` lit le texte; `json.loads()`
analyse ce texte pour en faire des objets Python. (`loads` signifie « charger à
partir d’une chaîne de caractères ».)

**JSON** (JavaScript Object Notation) est un format texte pour les données
structurées. Ce n’est pas un DataFrame et il ne ressemble pas nécessairement à une table.

| JSON | Python après décodage |
|---|---|
| objet : `{"id": 123}` | dictionnaire (`dict`) |
| tableau : `[1, 2, 3]` | liste (`list`) |
| chaîne de caractères / nombre | `str` / `int` ou `float` |
| `true`, `false`, `null` | `True`, `False`, `None` |

In [ ]:
game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == game_id
print("Python type:", type(game))
print("Top-level keys:", list(game.keys()))

In [ ]:
print("Game:", game["id"])
print("Date:", game["gameDate"])
print("Match:", game["awayTeam"]["abbrev"], "at", game["homeTeam"]["abbrev"])
print("Season:", game["season"])
print("Game type:", game["gameType"])

Le match complet est un dictionnaire. `game["plays"]` est une **liste de
dictionnaires décrivant les événements**. Un événement peut être une mise au jeu,
un tir, un but, une pénalité ou un début de période. Ce n’est pas nécessairement
un tir, et ces données ne décrivent pas chaque mouvement sur la glace.

Les crochets ont deux rôles ci-dessous : une chaîne de caractères sélectionne une
clé de dictionnaire, tandis qu’un entier sélectionne une position dans une liste.
Les positions dans les listes Python commencent à zéro.

In [ ]:
plays = game["plays"]
print("Type:", type(plays))
print("Number of events:", len(plays))
print("First event:\n", json.dumps(plays[0], indent=2))

In [ ]:
first_play = plays[0]
print("Event type:", first_play["typeDescKey"])
print("Period:", first_play["periodDescriptor"]["number"])
print("Time in period:", first_play["timeInPeriod"])
print("Optional details:", first_play.get("details", {}))

`periodDescriptor` est un dictionnaire **à l’intérieur** d’un dictionnaire
d’événement. Ce sont des données imbriquées. Les événements n’ont pas tous les
mêmes champs : un début de période peut ne pas avoir de `details` ni de coordonnées.
`get("details", {})` renvoie un dictionnaire vide si cette clé est absente, ce qui
permet de l’inspecter sans provoquer de `KeyError`. Cela n’invente pas de coordonnées.

**Exercice 1 (2 minutes) :** affichez le deuxième événement et son type.
Que signifie `plays[1]` ? Que tenterait de faire `game[1]` ?

In [ ]:
# Your turn: display the second event and print its event type.

<details>
<summary>Réponse de référence — à ouvrir après avoir essayé</summary>

```python
print(json.dumps(plays[1], indent=2))
print(plays[1]["typeDescKey"])
```

`plays[1]` sélectionne le deuxième élément d’une liste. `game[1]` cherche la clé
`1` dans le dictionnaire du match, mais cette clé n’existe pas.

</details>

## D. Des événements imbriqués à un DataFrame

Avant de créer une table, décidez ce que **représente une ligne** : ici, un
événement enregistré pour un match. Un DataFrame créé directement place les
dictionnaires imbriqués dans des cellules individuelles. Voyons d’abord ce résultat.

In [ ]:
nested_events = pd.DataFrame(plays)
display(nested_events[["eventId", "typeDescKey", "periodDescriptor"]].head())

`pd.json_normalize` aplatit les dictionnaires imbriqués en colonnes. Par exemple,
`periodDescriptor.number` devient un nom de colonne. Le point fait partie du nom;
on sélectionne donc cette colonne avec `events["periodDescriptor.number"]`.

Lorsqu’un champ est absent d’un événement, pandas indique que la valeur
correspondante est manquante. L’aplatissement change la **structure**, mais ne
décide pas quels champs ou événements sont utiles pour notre question.

In [ ]:
events = pd.json_normalize(plays)
print("Rows, columns:", events.shape)
print("Columns:", events.columns.tolist())
display(events.head())

In [ ]:
events.info()

In [ ]:
display(events["typeDescKey"].value_counts().rename("event_count").to_frame())
display(events[["details.xCoord", "details.yCoord"]].isna().sum().rename("missing_count"))

**Faites une pause et expliquez :** pourquoi la table contient-elle plus de lignes
que le nombre de buts ? Pourquoi un événement de début de période pourrait-il ne
pas avoir de coordonnées ? Remplacer ces coordonnées manquantes par zéro
préserverait-il le sens des données ?

L’index du DataFrame est une étiquette de ligne créée par pandas. Ce n’est **pas**
l’identifiant de l’événement de la NHL. Conservez `eventId`, puis combinez-le avec
l’identifiant du match pour identifier un événement parmi plusieurs matchs.

## E. Construire une petite table utile

Le jalon 1 commence avec les **tirs au but et les buts**. Dans ces données,
`shot-on-goal` et `goal` sont des catégories d’événements distinctes. Nous conservons
les deux; un but n’apparaît pas aussi dans une deuxième ligne `shot-on-goal`.
Les tirs manqués et bloqués sont d’autres catégories que nous excluons de cette
première table.

`.isin(...)` construit un masque booléen; `.loc[...]` sélectionne les lignes
correspondantes. `.copy()` nous donne une table distincte à modifier. Ce choix
dépend de la question que nous voulons étudier; il ne signifie pas que les autres
événements sont de mauvaises données.

In [ ]:
keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])
shots = events.loc[keep].copy()
print("All events:", len(events))
print("Retained shots and goals:", len(shots))
print("Other events left out:", len(events) - len(shots))

Sélectionnez quelques colonnes et donnez-leur des noms lisibles. `time_in_period`
est le temps écoulé **dans la période en cours**, pas depuis le début du match.
`period_type` permet de distinguer le temps réglementaire, la prolongation et les
tirs de barrage lorsque vous examinerez d’autres matchs.

`reindex(columns=...)` crée aussi les colonnes manquantes si un champ facultatif
est absent de toute la réponse. Nous conservons les valeurs inconnues comme
manquantes au lieu de les deviner. Les métadonnées du match sont ajoutées à chaque
ligne pour que la table reste compréhensible lorsque nous combinerons plusieurs matchs.

In [ ]:
column_names = {
    "eventId": "event_id",
    "periodDescriptor.number": "period",
    "periodDescriptor.periodType": "period_type",
    "timeInPeriod": "time_in_period",
    "typeDescKey": "event_type",
    "details.eventOwnerTeamId": "team_id",
    "details.xCoord": "x",
    "details.yCoord": "y",
    "details.shotType": "shot_type",
}
shots = shots.reindex(columns=list(column_names)).rename(columns=column_names)
shots["game_id"] = game["id"]
shots["season"] = game["season"]
shots["game_type"] = game["gameType"]
shots["is_goal"] = shots["event_type"].eq("goal")

Pour les événements conservés, l’équipe à laquelle l’événement est attribué est
celle qui effectue le tir. Utilisez les deux identifiants d’équipe fournis dans
les données du match pour ajouter des abréviations lisibles. `.map(...)` cherche
chaque identifiant dans un dictionnaire; un identifiant inconnu donne une valeur manquante.

`convert_dtypes()` choisit des types pandas capables de représenter les valeurs
manquantes. Par exemple, `Int64` (avec un I majuscule) stocke des entiers **et** un
marqueur de valeur manquante, contrairement à une colonne d’entiers NumPy ordinaire.
Les identifiants servent d’étiquettes; leur moyenne n’a pas de sens.

In [ ]:
team_names = {
    game["awayTeam"]["id"]: game["awayTeam"]["abbrev"],
    game["homeTeam"]["id"]: game["homeTeam"]["abbrev"],
}
shots["team"] = shots["team_id"].map(team_names)
shots = shots.convert_dtypes()
display(shots.head(10))

### Vérifier la table avant de l’utiliser

Une instruction `assert` arrête l’exécution si sa condition est fausse. Ces
vérifications permettent de repérer une perte accidentelle de lignes ou
d’identifiants d’événements, ou la création de doublons. Un événement est identifié
par **la combinaison** de l’identifiant du match et de celui de l’événement.
Ne supprimez pas les doublons sans comprendre pourquoi ils sont présents.

In [ ]:
assert len(events) == len(plays), "Flattening changed the number of events."
assert len(shots) == int(keep.sum()), "Cleaning changed the number of retained events."
assert shots[["game_id", "event_id"]].notna().all().all(), "Missing event identifier."
assert not shots.duplicated(["game_id", "event_id"]).any(), "Duplicate event identifiers."

print("Checks passed.")
display(shots.isna().sum().rename("missing_count").to_frame())

**Une valeur manquante n’est pas zéro.** Une coordonnée inconnue ne correspond pas
au centre de la patinoire, et une propriété inconnue n’est pas automatiquement
`False`. Conservez ces lignes dans `shots`. Si une analyse ultérieure nécessite
les deux coordonnées, créez un sous-ensemble distinct et indiquez combien de
lignes sont exclues. Pour ce match, ce nombre peut être zéro : une vérification
reste utile même lorsqu’elle ne trouve aucun problème.

In [ ]:
shots_with_coordinates = shots.dropna(subset=["x", "y"])
print("Rows excluded from a coordinate-based analysis:", len(shots) - len(shots_with_coordinates))
print("Rows kept in the full shots table:", len(shots))

### Lire un premier résumé

Avec des valeurs booléennes, `sum()` compte les valeurs `True`. `size` compte les
lignes. Ici, `shot_events` comprend les tirs conservés qui n’ont pas donné de but
**et** les buts. Ce petit résumé nous aide à vérifier les données; un seul match
ne suffit pas pour classer les équipes ou les joueurs.

In [ ]:
summary = shots.groupby("team", dropna=False).agg(
    shot_events=("event_id", "size"),
    goals=("is_goal", "sum"),
)
display(summary)

**Exercice 2 (3 minutes) :** créez une table contenant uniquement les buts de la
première période. Affichez `team`, `time_in_period` et `shot_type`. Combien de
lignes contient-elle ?

Indice : combinez deux conditions booléennes avec `&`, en entourant chacune de parenthèses.

In [ ]:
# Your turn: filter shots, then select the three requested columns.

<details>
<summary>Réponse de référence — à ouvrir après avoir essayé</summary>

```python
first_period_goals = shots.loc[
    (shots["period"] == 1) & shots["is_goal"],
    ["team", "time_in_period", "shot_type"],
]
display(first_period_goals)
print("Number of first-period goals:", len(first_period_goals))
```

</details>

## F. Sauvegarder la table obtenue et relier les étapes

Le **JSON brut** conserve les champs d’origine. Le **CSV traité** stocke les lignes
et colonnes que nous avons choisies. Conservez les deux : vous pouvez reconstruire
le CSV à partir du JSON si vos règles de nettoyage changent. `index=False` évite
d’écrire les étiquettes de lignes de pandas dans une colonne supplémentaire.

Le CSV est pratique pour consulter et partager les données, mais il ne conserve
pas tous les types pandas. Lorsque vous le rechargez, vérifiez à nouveau les types,
en particulier ceux des identifiants et des valeurs manquantes. Le JSON brut
reste l’entrée de notre processus de nettoyage.

In [ ]:
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)
csv_path = processed_dir / f"{game_id}_shots.csv"
shots.to_csv(csv_path, index=False)
print("Saved processed table:", csv_path.resolve())